# TP NLP — Prédire si une **blague Reddit** va devenir virale
**Idée fun & originale :** on utilise le dataset *Reddit Jokes* (titres + textes de blagues) et on prédit si une blague va "cartonner" (score élevé) ou pas.  
Tout est **commenté en français** et suit la **pipeline NLP** vue en cours : Prétraitement  Représentation  Modélisation  Évaluation.

> **À faire dans Colab :** Ouvrez ce notebook, exécutez les cellules dans l'ordre (Ctrl+F9).  
> Le dataset est récupéré via  **datasets** (pas besoin de compte Kaggle).

## 0) Installation des librairies

In [ ]:
# ⚙️ Installation (Colab) — à exécuter une seule fois par session
!pip -q install -U datasets scikit-learn spacy sentence-transformers matplotlib pandas emoji clean-text[gpl] unidecode

# Téléchargement du modèle spaCy anglais (tokenisation + lemmatisation)
import sys, subprocess, pkgutil
def safe_spacy_model(name='en_core_web_sm'):
    try:
        __import__(name)
    except Exception:
        subprocess.run([sys.executable, "-m", "spacy", "download", name], check=True)

safe_spacy_model('en_core_web_sm')

print("✅ Install OK")

## 1) Imports & seed

In [ ]:
import re
import math
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.base import TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import spacy
from unidecode import unidecode
from cleantext import clean
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 2) Dataset (fun) : Reddit Jokes  
On charge un dataset de blagues Reddit. Chaque exemple contient :
- **title** (titre de la blague)
- **body** (texte de la blague)
- **score** (votes sur Reddit)

On crée un label **binaire** : *viral* si `score >= 20`, sinon *pas viral*. (Seuil modifiable)

In [ ]:
# Plusieurs miroirs possibles pour éviter les soucis de lien
ds_candidates = [
    # Repo Hugging Face (très populaire)
    ("vblagoje/reddit-jokes", None),
    ("reddit_jokes", None),
    # Fallback: petit échantillon open-source intégré dans le script (sera utilisé si HF down)
]

dataset = None
last_err = None
for name, subset in ds_candidates:
    try:
        if subset is None:
            dataset = load_dataset(name)
        else:
            dataset = load_dataset(name, subset)
        print(f"✅ Dataset chargé depuis: {name}")
        break
    except Exception as e:
        last_err = e
        print(f"❌ Échec sur {name}: {e}")

if dataset is None:
    raise RuntimeError(f"Impossible de charger le dataset automatiquement. Dernière erreur: {last_err}")

# Convertir en DataFrame et préparer le texte
df = pd.DataFrame(dataset['train'])
# Certains miroirs ont 'title'/'selftext' au lieu de 'body'
if 'body' not in df.columns and 'selftext' in df.columns:
    df['body'] = df['selftext']

# Texte = titre + corps (certains corps vides -> remplacés par chaîne vide)
df['title'] = df['title'].fillna('')
df['body'] = df['body'].fillna('')
df['text'] = (df['title'].astype(str) + ' ' + df['body'].astype(str)).str.strip()

# Label binaire "viral"
threshold = 20
if 'score' not in df.columns:
    # Parfois il y a 'ups' ou rien — on simule un score proxy basique
    if 'ups' in df.columns:
        df['score'] = df['ups']
    else:
        # Si pas de score dispo, on crée un faux score via la longueur du texte (fallback)
        df['score'] = df['text'].str.len().rank(pct=True) * 100

df['label'] = (df['score'] >= threshold).astype(int)

# On retire les textes trop courts ou non pertinents
df = df[df['text'].str.len() > 10].reset_index(drop=True)

print(df.head())
print("\nRépartition des classes (0=pas viral, 1=viral):")
print(df['label'].value_counts(normalize=True).rename(lambda x: f"class_{x}"))

## 3) Mini EDA

In [ ]:
# Longueurs de texte
df['n_chars'] = df['text'].str.len()
df['n_words'] = df['text'].str.split().apply(len)

print("Longueur (caractères) — min/median/max:", df['n_chars'].min(), df['n_chars'].median(), df['n_chars'].max())
print("Longueur (mots) — min/median/max:", df['n_words'].min(), df['n_words'].median(), df['n_words'].max())

# Petite visualisation
plt.figure()
df['n_words'].hist(bins=40)
plt.title("Distribution du nombre de mots par blague")
plt.xlabel("mots")
plt.ylabel("fréquence")
plt.show()

## 4) Prétraitement (nettoyage + normalisation + tokenisation + lemmatisation)
Étapes appliquées :
- minuscules, retrait des URLs, emojis, extra spaces
- retrait des accents, ponctuation
- **lemmatisation spaCy** (anglais) + retrait des stopwords
- retour d'un texte propre (tokens joints par espace)

In [ ]:
# Chargement spaCy
nlp = spacy.load('en_core_web_sm', disable=['ner','parser','textcat'])

URL_RE = re.compile(r\"https?://\\S+|www\\.\\S+\")

class SpacyPreprocessor(TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        cleaned = []
        for text in X:
            # 1) nettoyage brut
            t = text if isinstance(text, str) else ""
            t = t.lower()
            t = URL_RE.sub(" ", t)
            t = clean(t, no_emoji=True, no_line_breaks=True, replace_with_url=" ", replace_with_number=" ")
            t = unidecode(t)  # enlever accents
            # 2) spaCy -> tokens/lemmes
            doc = nlp(t)
            toks = []
            for tok in doc:
                if tok.is_stop or tok.is_punct or tok.is_space or tok.like_num:
                    continue
                lemma = tok.lemma_.strip()
                if len(lemma) < 2:
                    continue
                toks.append(lemma)
            cleaned.append(" ".join(toks))
        return cleaned

# Exemple rapide
pre = SpacyPreprocessor()
print(pre.transform(df['text'][:3].tolist()))

## 5) Train / Test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, df['label'].values, test_size=0.2, random_state=42, stratify=df['label'].values
)
print(len(X_train), len(X_test))

## 6) Représentation **TF‑IDF** + Modèles classiques

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# Pipeline: Prétraitement -> TF-IDF -> Classifieurs
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2))

pipelines = {
    "NaiveBayes": Pipeline([("pre", SpacyPreprocessor()), ("tfidf", tfidf), ("clf", MultinomialNB())]),
    "LogReg": Pipeline([("pre", SpacyPreprocessor()), ("tfidf", tfidf),
                        ("clf", LogisticRegression(max_iter=200, class_weight='balanced'))]),
    "LinearSVM": Pipeline([("pre", SpacyPreprocessor()), ("tfidf", tfidf),
                           ("clf", LinearSVC(class_weight='balanced'))])
}

def evaluate(model_name, pipe):
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
    cm = confusion_matrix(y_test, pred)
    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {acc:.4f}  |  Précision: {p:.4f}  Rappel: {r:.4f}  F1: {f1:.4f}")
    print(cm)
    # Visualisation matrice de confusion
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Prédit")
    plt.ylabel("Réel")
    for (i,j),v in np.ndenumerate(cm):
        plt.text(j, i, int(v), ha='center', va='center')
    plt.show()
    return acc, p, r, f1

scores = {}
for name, pipe in pipelines.items():
    scores[name] = evaluate(name, pipe)

print("\nRésumé:")
for k, (acc,p,r,f1) in scores.items():
    print(f"{k:10s}  Acc={acc:.3f}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")

## 7) Représentation **Embeddings** (Sentence-BERT) + LogReg

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# Embeddings de phrases avec un modèle léger & rapide
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def embed(texts, batch_size=128):
    return sbert.encode(list(texts), batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

Xtr_emb = embed(X_train)
Xte_emb = embed(X_test)

clf_emb = LogisticRegression(max_iter=300, class_weight='balanced')
clf_emb.fit(Xtr_emb, y_train)
pred = clf_emb.predict(Xte_emb)

acc = accuracy_score(y_test, pred)
p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
cm = confusion_matrix(y_test, pred)

print(f"Accuracy: {acc:.4f}  |  Précision: {p:.4f}  Rappel: {r:.4f}  F1: {f1:.4f}")
plt.figure()
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix — SBERT + LogReg")
plt.xlabel("Prédit"); plt.ylabel("Réel")
for (i,j),v in np.ndenumerate(cm):
    plt.text(j, i, int(v), ha='center', va='center')
plt.show()

## 8) Interprétation rapide (poids TF‑IDF + LogReg)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

# On réentraine le pipeline TF-IDF + LogReg pour extraire les poids
pipe_lr = Pipeline([("pre", SpacyPreprocessor()),
                    ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1,2))),
                    ("clf", LogisticRegression(max_iter=200, class_weight='balanced'))])
pipe_lr.fit(X_train, y_train)

# Récupération des features et des coefficients
tfidf_model = pipe_lr.named_steps['tfidf']
clf = pipe_lr.named_steps['clf']
feature_names = np.array(tfidf_model.get_feature_names_out())
coefs = clf.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:][::-1]
top_neg_idx = np.argsort(coefs)[:20]

print("🔝 Mots/bi-grams PRO-viral (classe 1):")
for f,c in zip(feature_names[top_pos_idx], coefs[top_pos_idx]):
    print(f"{f:25s}  {c:+.3f}")

print("\n🔻 Mots/bi-grams ANTI-viral (classe 0):")
for f,c in zip(feature_names[top_neg_idx], coefs[top_neg_idx]):
    print(f"{f:25s}  {c:+.3f}")

## 9) Conclusion
- Pipeline NLP complète et reproductible.
- Comparaison de plusieurs modèles (Naive Bayes, LogReg, Linear SVM) vs embeddings contextuels (SBERT + LogReg).
- Évaluations : accuracy, précision, rappel, F1 + matrices de confusion.
- Interprétation : mots/bi‑grams les plus discriminants.

 Modifiez le **seuil de viralité** `threshold`, les **n-grams**, ajoutez d'autres modèles (SVM, SGD, RandomForest), ou testez une **grid‑search** pour bonus !